Filtering. The first model building step is largely the same

In [1]:
import jax.numpy as jnp
import jax.random as jr
import numpyro
import numpyro.distributions as dist
from numpyro.infer import Predictive

import dynestyx as dsx
from dynestyx import DiscreteTimeSimulator, DynamicalModel

# for convenience, we can define "fixed" things in the model outside of it.
# this is not required, but it helps keep the model clean.
state_dim = 2
observation_dim = 1
control_dim = 1

# Create the known matrices B, C
B = jnp.eye(state_dim, control_dim)
C = jnp.eye(observation_dim, state_dim)

# create the initial condition as a distribution
initial_condition = dist.MultivariateNormal(jnp.zeros(state_dim), jnp.eye(state_dim))

#Hard coded variances for noise, but could also be unknown in other contexts
def lti_model(
    sigma_obs=0.1,
    sigma_process=0.1,
    obs_times=None,
    obs_values=None,
    ctrl_times=None,
    ctrl_values=None,
    predict_times=None,
):
    # sample the unknown parameter
    rho = numpyro.sample("rho", dist.Uniform(-0.5, 0.5))
    A = jnp.array([[0, 0.3], [rho, -0.2]])

    # create the state evolution as a callable mapping to a distribution
    # Crucially, this depends on A, which depends on rho, which is unknown.
    # Thus, the state evolution MUST be defined within `lti_model`, not outside.
    state_evolution = lambda x, u, t_now, t_next: dist.MultivariateNormal(
        A @ x + B @ u, sigma_process**2 * jnp.eye(state_dim)
    )

    # create the observation model as a callable mapping to a distribution
    observation_model = lambda x, u, t: dist.MultivariateNormal(
        C @ x, sigma_obs**2 * jnp.eye(observation_dim)
    )

    # create the dynamical model
    dynamics = DynamicalModel(
        control_dim=control_dim,
        initial_condition=initial_condition,
        state_evolution=state_evolution,
        observation_model=observation_model,
    )

    # sample from the dynamical model
    return dsx.sample(
        "f",
        dynamics,
        obs_times=obs_times,
        obs_values=obs_values,
        ctrl_times=ctrl_times,
        ctrl_values=ctrl_values,
        predict_times=predict_times,
    )

Generating Data with rho=0.3

The resulting obs_values has both the leading n_simulations and num_samples axis that we found in the previous section.

In [2]:
# create a synthetic control sequence as i.i.d. Gaussians
obs_times = jnp.arange(0.0, 100.0, 1.0)  # T=100 steps
ctrl_times = obs_times  # same times for controls
ctrl_values = jr.normal(jr.PRNGKey(0), (len(ctrl_times), control_dim))

rho_true = 0.3


def make_data(sigma_obs=0.1, sigma_process=0.1):
    predictive = Predictive(
        lti_model,
        params={"rho": jnp.array(rho_true)},
        num_samples=1,
        exclude_deterministic=False,
    )
    with DiscreteTimeSimulator():
        pred = predictive(
            rng_key=jr.PRNGKey(0),
            sigma_obs=sigma_obs,
            sigma_process=sigma_process,
            predict_times=obs_times,
            ctrl_times=ctrl_times,
            ctrl_values=ctrl_values,
        )
    print("make_data shapes:", pred["f_times"].shape, pred["f_observations"].shape)
    # Expected: f_observations has shape (1, 1, T, obs_dim)
    obs_values = pred["f_observations"][0, 0, :, :]
    return obs_times, obs_values, ctrl_times, ctrl_values


obs_times, obs_values, ctrl_times, ctrl_values = make_data(sigma_obs=0.1, sigma_process=0.1)

make_data shapes: (1, 1, 100) (1, 1, 100, 1)


Now we introduce the filtering algorithm, a function that computes the marginal log likelihood for a given parameter rho

In [ ]:
from dynestyx import Filter
from dynestyx.inference.filters import EnKFConfig, EKFConfig, PFConfig

def get_mll(rho, filter_config):
    """Evaluate model at fixed params via Predictive and return the MLL (deterministic site)."""

    with Filter(filter_config):
        predictive = Predictive(
            lti_model,
            params={"rho": rho},
            num_samples=1,
            exclude_deterministic=False,
        )
        mll = predictive(jr.PRNGKey(0), obs_times=obs_times, obs_values=obs_values, ctrl_times=ctrl_times, ctrl_values=ctrl_values)["f_marginal_loglik"]

    return mll


get_mll(0.3, EKFConfig())

Naturally, we vary the computation of the log likelihood over different values of rho as below

In [ ]:
from jax import vmap

# Profile over values of rho, keeping other parameters at their true values:
rho_grid = jnp.linspace(-0.8, 0.8, 50)

mll_enkf = vmap(lambda p: get_mll(p, filter_config=EnKFConfig(n_particles=100)))(rho_grid)
mll_kf = vmap(lambda p: get_mll(p, filter_config=EKFConfig()))(rho_grid)
mll_pf = vmap(lambda p: get_mll(p, filter_config=PFConfig(n_particles=1000)))(rho_grid)

Plot all mll's

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(rho_grid, mll_pf, "o-", color="C0", label="Profile likelihood (PF with 1000 particles)")
ax.plot(rho_grid, mll_kf, "o-", color="C1", label="Profile likelihood (Taylor KF)")
ax.plot(rho_grid, mll_enkf, "o-", color="C2", label="Profile likelihood (EnKF with 100 particles)")
ax.axvline(rho_true, color="k", linestyle="--", label=r"ρtrue")
ax.set_xlabel(r"ρ", fontsize=12)
ax.set_ylabel(r"log⁡p(y1:T∣ρ)", fontsize=12)
ax.set_title("Profile likelihood (filtering algorithms)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()